In [ ]:
%pip install -q duckdb pyarrow "fg-data-profiling[notebook]" "setuptools<82"

In [ ]:
from pathlib import Path
import shutil

import duckdb
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/TLC-Trip-Record-Dat")
SOURCE_DATA_DIR = Path("/content/drive/MyDrive/data")
DATA_DIR = Path("/content/data")
OUTPUT_DIR = PROJECT_DIR / "output"
TEMP_DIR = Path("/content/tlc_duckdb")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

for source in sorted(SOURCE_DATA_DIR.glob("*.parquet")):
    destination = DATA_DIR / source.name
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)

PARQUET_GLOB = str(DATA_DIR / "*.parquet")
PARQUET_GLOB_SQL = PARQUET_GLOB.replace("'", "''")
TEMP_DIR_SQL = str(TEMP_DIR).replace("'", "''")

In [ ]:
parquet_files = sorted(DATA_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"В {DATA_DIR} не найдены Parquet-файлы")

file_inventory = pd.DataFrame(
    {
        "file": path.name,
        "rows": pq.read_metadata(path).num_rows,
        "size_gb": path.stat().st_size / 1_000_000_000,
    }
    for path in parquet_files
)

display(file_inventory)
print(f"Файлов: {len(file_inventory)}")
print(f"Строк: {file_inventory['rows'].sum():,}")
print(f"Размер на диске: {file_inventory['size_gb'].sum():.2f} GB")

In [ ]:
con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")
con.execute("SET threads = 4")
con.execute(f"SET temp_directory = '{TEMP_DIR_SQL}'")

con.execute(
    f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT *
    FROM read_parquet(
        '{PARQUET_GLOB_SQL}',
        filename = true
    )
    """
)

In [ ]:
schema = con.sql("DESCRIBE trips").df()
schema

In [ ]:
quality = con.sql(
    """
    WITH checks AS (
        SELECT
            filename,
            request_datetime,
            pickup_datetime,
            dropoff_datetime,
            date_diff('second', request_datetime, pickup_datetime) AS wait_seconds,
            trip_time,
            trip_miles
        FROM trips
    )
    SELECT
        count(*) AS rows,
        count(DISTINCT filename) AS files,
        min(pickup_datetime) AS first_pickup,
        max(pickup_datetime) AS last_pickup,
        count(*) FILTER (WHERE request_datetime IS NULL) AS missing_request_datetime,
        count(*) FILTER (WHERE pickup_datetime IS NULL) AS missing_pickup_datetime,
        count(*) FILTER (WHERE dropoff_datetime IS NULL) AS missing_dropoff_datetime,
        count(*) FILTER (WHERE wait_seconds < 0 OR wait_seconds > 7200) AS invalid_wait,
        count(*) FILTER (WHERE trip_time <= 0) AS non_positive_trip_time,
        count(*) FILTER (WHERE trip_miles < 0) AS negative_trip_miles
    FROM checks
    """
).df()

quality.T

Пример продуктового вопроса: где и когда пассажиры дольше всего ждут машину? Единица анализа — завершённая поездка. Основные метрики — число поездок, медиана и p90 ожидания; стоимость и выплата водителю используются как дополнительные показатели.

Расчёт выполняется по всему датасету, но в pandas загружается только агрегированный результат.

In [ ]:
metrics = con.sql(
    """
    WITH clean_trips AS (
        SELECT
            date_trunc('month', pickup_datetime) AS month,
            hvfhs_license_num AS provider,
            PULocationID AS pickup_zone,
            extract(hour FROM pickup_datetime) AS pickup_hour,
            date_diff('second', request_datetime, pickup_datetime) AS wait_seconds,
            trip_miles,
            trip_time,
            base_passenger_fare,
            driver_pay
        FROM trips
        WHERE request_datetime IS NOT NULL
          AND pickup_datetime IS NOT NULL
          AND dropoff_datetime IS NOT NULL
    )
    SELECT
        month,
        provider,
        pickup_zone,
        pickup_hour,
        count(*) AS trips,
        approx_quantile(wait_seconds, 0.50) AS median_wait_seconds,
        approx_quantile(wait_seconds, 0.90) AS p90_wait_seconds,
        avg(trip_miles) AS avg_trip_miles,
        avg(trip_time) / 60.0 AS avg_trip_minutes,
        avg(base_passenger_fare) AS avg_base_fare,
        avg(driver_pay) AS avg_driver_pay
    FROM clean_trips
    WHERE wait_seconds BETWEEN 0 AND 7200
      AND trip_time > 0
      AND trip_miles >= 0
    GROUP BY 1, 2, 3, 4
    ORDER BY 1, 2, 3, 4
    """
).df()

metrics.info()
metrics.head()

`ProfileReport` нельзя строить на полном датасете размером 65+ GB. DuckDB формирует reservoir-выборку с фиксированным seed, и только она преобразуется в pandas DataFrame. Эта ячейка также читает весь набор файлов, поэтому выборку имеет смысл сохранить и переиспользовать.

In [ ]:
df_sample = con.sql(
    """
    SELECT * EXCLUDE (filename)
    FROM trips
    USING SAMPLE reservoir(100000 ROWS)
    REPEATABLE (42)
    """
).df()

SAMPLE_PATH = OUTPUT_DIR / "tlc_sample.parquet"
df_sample.to_parquet(SAMPLE_PATH, index=False)

df_sample.info()
print(f"Выборка сохранена: {SAMPLE_PATH}")